# 🎬 Geração de Vídeo com Prompt + Imagem (AnimateDiff)

In [ ]:
!pip install diffusers accelerate einops transformers ffmpeg-python gradio animate-diff --quiet

In [ ]:
import os
import torch
import ffmpeg
from PIL import Image
import gradio as gr
from diffusers import DiffusionPipeline

os.makedirs('inputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)


In [ ]:
def generate_video(image_path, prompt, duration=2, fps=14):
    try:
        print(f"▶️ Iniciando geração | prompt: '{prompt}' | imagem: {image_path}")
        model_id = 'animate-diff'  # placeholder correto
        pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
        pipe = pipe.to('cuda')

        image = Image.open(image_path).convert('RGB')
        image = image.resize((512, 512))  # ajuste conforme o modelo

        output = pipe(
            prompt=prompt,
            init_image=image,
            num_inference_steps=50,
            num_frames=int(duration * fps)
        )

        frames = output.frames if hasattr(output, 'frames') else output['frames']
        temp = 'outputs/temp_frames'
        os.makedirs(temp, exist_ok=True)
        for i, f in enumerate(frames):
            f.save(f"{temp}/frame_{i:04d}.png")

        out = f"outputs/{os.path.basename(image_path).split('.')[0]}_gen.mp4"
        ffmpeg.input(f"{temp}/frame_%04d.png", framerate=fps)\
              .output(out, vcodec='libx264', pix_fmt='yuv420p')\
              .run(overwrite_output=True)

        print('✅ Vídeo gerado em', out)
        return out
    except Exception as e:
        print('❌ Falha na geração:', e)
        return None


In [ ]:
def video_app(img, prompt, duration, fps):
    out = generate_video(img, prompt, duration, fps)
    if out is None:
        raise gr.Error('❌ Erro. Veja o console para detalhes.')
    return out


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('## 🌅 Vídeo Cinematográfico via Imagem + Prompt')
    with gr.Row():
        img = gr.Image(label='Imagem de Entrada', type='filepath')
        prm = gr.Textbox(label='Prompt (ex: "waves moving, sand blowing, sunset")', value='')
    with gr.Row():
        dur = gr.Slider(label='Duração (s)', minimum=1, maximum=10, step=0.5, value=2)
        fps = gr.Slider(label='FPS', minimum=6, maximum=30, step=1, value=14)
    btn = gr.Button('Gerar Vídeo')
    vid = gr.Video(label='Resultado')
    btn.click(fn=video_app, inputs=[img, prm, dur, fps], outputs=vid)
    demo.launch(share=True, debug=True)


In [ ]:
# Para baixar automaticamente:
# from google.colab import files
# files.download(out)
